In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from topostats.tracing.dnatracing import trace_image, trace_grain, pad_bounding_box, crop_array
from topostats.plottingfuncs import Images

NOTEBOOK_DIR = Path.cwd()
BASE_DIR = NOTEBOOK_DIR.parent
TEST_DIR = BASE_DIR / "tests"
TMP_DIR = BASE_DIR / "tmp" / "plot_traces"
RESOURCES_DIR = TEST_DIR / "resources"
# Load Test images
LINEAR_IMAGE = np.load(RESOURCES_DIR / "dnatracing_image_linear.npy")
LINEAR_MASK = np.load(RESOURCES_DIR / "dnatracing_mask_linear.npy")
CIRCULAR_IMAGE = np.load(RESOURCES_DIR / "dnatracing_image_circular.npy")
CIRCULAR_MASK = np.load(RESOURCES_DIR / "dnatracing_mask_circular.npy")
PADDED_LINEAR_IMAGE = np.pad(LINEAR_IMAGE, ((7, 6), (4, 4)))
PADDED_LINEAR_MASK = np.pad(LINEAR_MASK, ((7, 6), (4, 4)))
CIRCULAR_MASK = np.where(CIRCULAR_MASK == 1, 2, CIRCULAR_MASK)
MULTIGRAIN_IMAGE = np.concatenate((PADDED_LINEAR_IMAGE, CIRCULAR_IMAGE), axis=0)
MULTIGRAIN_MASK = np.concatenate((PADDED_LINEAR_MASK, CIRCULAR_MASK), axis=0)
PAD_WIDTH = 30
PIXEL_SIZE = 0.4940029296875
MIN_SKELETON_SIZE = 10
PAD_WIDTH = 30

print(f"Linear Image shape     : {LINEAR_IMAGE.shape}")
print(f"Circular Image shape   : {CIRCULAR_IMAGE.shape}")
print(f"Multigrain Image shape : {MULTIGRAIN_IMAGE.shape}")

In [ ]:
plt.imshow(LINEAR_IMAGE)
plt.show()
plt.imshow(CIRCULAR_IMAGE)
plt.show()
plt.imshow(MULTIGRAIN_IMAGE)
plt.show()

In [ ]:
linear_trace = trace_grain(
    LINEAR_IMAGE,
    LINEAR_MASK,
    pixel_to_nm_scaling=PIXEL_SIZE,
    filename="linear",
    min_skeleton_size=MIN_SKELETON_SIZE,
    skeletonisation_method="topostats",
)
circular_trace = trace_grain(
    CIRCULAR_IMAGE,
    CIRCULAR_MASK,
    pixel_to_nm_scaling=PIXEL_SIZE,
    filename="circular",
    min_skeleton_size=MIN_SKELETON_SIZE,
    skeletonisation_method="topostats",
)
multigrain_trace = trace_image(
    MULTIGRAIN_IMAGE,
    MULTIGRAIN_MASK,
    pixel_to_nm_scaling=PIXEL_SIZE,
    filename="multigrain",
    min_skeleton_size=MIN_SKELETON_SIZE,
    skeletonisation_method="topostats",
)

In [ ]:
def make_mask(image: np.ndarray, trace: np.ndarray) -> np.ndarray:
    """Make a mask for returned co-ordinates"""
    mask = np.zeros(image.shape)
    for coord in trace:
        mask[coord[0], coord[1]] = 1
    return mask


LINEAR_TRACE = make_mask(LINEAR_IMAGE, linear_trace["ordered_trace"])
plt.imshow(LINEAR_TRACE)
plt.show()
CIRCULAR_TRACE = make_mask(CIRCULAR_IMAGE, circular_trace["ordered_trace"])
plt.imshow(CIRCULAR_TRACE)
plt.show()
MULTIGRAIN_TRACE = make_mask(MULTIGRAIN_IMAGE, circular_trace["ordered_trace"])
plt.imshow(MULTIGRAIN_TRACE)
plt.show()

# mask.sum()

In [ ]:
linear_trace

# Remake original cropped images

In [ ]:
import pickle as pkl

grain3 = BASE_DIR / "tmp" / "individual_grains" / "grain_3.pkl"
grain9 = BASE_DIR / "tmp" / "individual_grains" / "grain_9.pkl"


def quick_plot(img_path):
    with img_path.open("rb") as input:
        single_grain = pkl.load(input)
    image = single_grain["original"]
    mask = single_grain["mask"]
    plt.imshow(image)
    plt.show()
    plt.imshow(mask)
    plt.show()


quick_plot(grain3)

In [ ]:
quick_plot(grain9)

In [ ]:
linear_image = np.load(BASE_DIR / "tests" / "resources" / "dnatracing_image_linear.npy")
plt.imshow(linear_image)
plt.show()
linear_mask = np.load(BASE_DIR / "tests" / "resources" / "dnatracing_mask_linear.npy")
plt.imshow(linear_mask)
plt.show()
linear_image.shape

In [ ]:
circular_image = np.load(BASE_DIR / "tests" / "resources" / "dnatracing_image_circular.npy")
plt.imshow(circular_image)
plt.show()
circular_mask = np.load(BASE_DIR / "tests" / "resources" / "dnatracing_mask_circular.npy")
plt.imshow(circular_mask)
plt.show()
circular_image.shape

In [ ]:
np.asarray(circular_image.shape) - np.asarray(linear_image.shape)

In [ ]:
multigrain_image = np.load(BASE_DIR / "tests" / "resources" / "dnatracing_multigrain_image.npy")
plt.imshow(multigrain_image)
plt.show()
multigrain_mask = np.load(BASE_DIR / "tests" / "resources" / "dnatracing_multigrain_mask.npy")
plt.imshow(multigrain_mask)
plt.show()
multigrain_image.shape

In [ ]:
import pickle as pkl

test = RESOURCES_DIR / "multigrain_results.pkl"
with test.open("rb") as f:
    results = pkl.load(f)

In [ ]:
multigrain_trace = trace_image(
    MULTIGRAIN_IMAGE,
    MULTIGRAIN_MASK,
    pixel_to_nm_scaling=PIXEL_SIZE,
    filename="multigrain_pad30",
    min_skeleton_size=MIN_SKELETON_SIZE,
    pad_width=30,
    skeletonisation_method="topostats",
)
plt.imshow(multigrain_trace["image_trace"])
plt.show()
plt.imshow(MULTIGRAIN_IMAGE)
plt.show()
fig, ax = Images(
    data=MULTIGRAIN_IMAGE,
    masked_array=multigrain_trace["image_trace"],
    output_dir=TMP_DIR,
    filename="multigrain_traced30",
    image_set="all",
    mask_cmap="viridis",
    dpi=1000,
).plot_and_save()
# fig

In [ ]:
minic = np.load(BASE_DIR / "output_main/tests/resources/processed/minicircle_height_thresholded.npy")

In [ ]:
minic.shape
minic_below = minic[210:325, 575:704]
plt.imshow(minic_below)
plt.show()